In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix
)

from xgboost import XGBClassifier


TRAIN_PATH = "../data/processed/train_table_train_2015_2024.parquet"
TEST_PATH  = "../data/processed/train_table_test_2025.parquet"
OUT_DIR    = "../models"
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42


train_table = pd.read_parquet(TRAIN_PATH)
test_table  = pd.read_parquet(TEST_PATH)

train_table["time_bin"] = pd.to_datetime(train_table["time_bin"])
test_table["time_bin"]  = pd.to_datetime(test_table["time_bin"])

train_df = train_table[train_table["time_bin"].dt.year <= 2023]
val_df   = train_table[train_table["time_bin"].dt.year == 2024]
test_df  = test_table

print("Train/Val/Test:", train_df.shape, val_df.shape, test_df.shape)


drop_cols = ["y", "time_bin"]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df["y"].astype(int)

X_val = val_df.drop(columns=drop_cols)
y_val = val_df["y"].astype(int)

X_test = test_df.drop(columns=drop_cols)
y_test = test_df["y"].astype(int)

cat_cols = ["cell_id"]
num_cols = [c for c in X_train.columns if c not in cat_cols]


preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), cat_cols),
        ("num", "passthrough", num_cols),
    ]
)

X_train_enc = preprocess.fit_transform(X_train)
X_val_enc   = preprocess.transform(X_val)
X_test_enc  = preprocess.transform(X_test)


xgb = XGBClassifier(
    n_estimators=1200,       
    learning_rate=0.03,
    max_depth=6,

    subsample=0.8,
    colsample_bytree=0.8,

    min_child_weight=5,

    reg_alpha=0.0,
    reg_lambda=1.0,

    gamma=0.0,

    objective="binary:logistic",
    eval_metric="auc",

    tree_method="hist",       
    random_state=RANDOM_STATE,
    n_jobs=-1
)


xgb.fit(X_train_enc, y_train)


def eval_split(name, X, y):
    proba = xgb.predict_proba(X)[:, 1]
    pred  = (proba >= 0.5).astype(int)

    auc = roc_auc_score(y, proba)
    ap  = average_precision_score(y, proba)

    print(f"\n[{name}] AUC={auc:.4f}  AP={ap:.4f}")
    print("Confusion:\n", confusion_matrix(y, pred))
    print(classification_report(y, pred, digits=4))

eval_split("VAL(2024)", X_val_enc, y_val)
eval_split("TEST(2025)", X_test_enc, y_test)


bundle = {
    "preprocess": preprocess,
    "model": xgb
}

joblib.dump(bundle, os.path.join(OUT_DIR, "xgb_grid.pkl"))
print("\nSaved:", os.path.join(OUT_DIR, "xgb_grid.pkl"))

Train/Val/Test: (292810, 15) (33019, 15) (31773, 15)

[VAL(2024)] AUC=0.8539  AP=0.8975
Confusion:
 [[ 8807  4281]
 [ 3054 16877]]
              precision    recall  f1-score   support

           0     0.7425    0.6729    0.7060     13088
           1     0.7977    0.8468    0.8215     19931

    accuracy                         0.7779     33019
   macro avg     0.7701    0.7598    0.7637     33019
weighted avg     0.7758    0.7779    0.7757     33019


[TEST(2025)] AUC=0.8515  AP=0.8817
Confusion:
 [[ 9376  4278]
 [ 2914 15205]]
              precision    recall  f1-score   support

           0     0.7629    0.6867    0.7228     13654
           1     0.7804    0.8392    0.8087     18119

    accuracy                         0.7736     31773
   macro avg     0.7717    0.7629    0.7658     31773
weighted avg     0.7729    0.7736    0.7718     31773


Saved: ../models/xgb_grid.pkl
